# RR breaking point and workload rates

This notebook estimates the request arrival rate where round-robin (RR) stops keeping up for each FairRoute workload. It inspects the pilot trace shapes, then measures the RR response in the existing experiment outputs. It prints a suggested `LAM_W1` … `LAM_W5` block for `scripts/build_fairroute_suite.sh`.

**Data status:** the pilot JSONL traces are in `workloads/fairroute/pilot/`. Direct RR pilot result CSVs were not present when this notebook was prepared. Until they are generated and placed in `outputs/fairroute/pilot/<workload>/<pilot-cell>/trace_policy_RR.csv`, the notebook uses the existing RR main-suite summary as the observed response curve. The resulting values are estimates, and the notebook labels their source.

The knee rule is deliberately simple and inspectable: choose the tested arrival rate immediately below the first clear queue-delay escalation, using both p99 queue delay and completion throughput. W4 is session-arrival rate; all other workloads use requests/s.


In [ ]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "workloads/fairroute/pilot").exists():
    ROOT = ROOT.parent
PILOT_DIR = ROOT / "workloads/fairroute/pilot"
PILOT_RESULTS = ROOT / "outputs/fairroute/pilot"
SUMMARY_CSV = ROOT / "notebooks/outputs/fairroute_full_summary.csv"


## Inspect pilot traces

In [ ]:
def inspect_trace(path):
    rows = []
    with path.open() as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                rows.append((int(r["arrival_time_ns"]), int(r["input_toks"]), int(r["output_toks"]), int(r["client_id"])))
    if not rows:
        return None
    a = np.asarray(rows, dtype=np.int64)
    span_s = max((a[:, 0].max() - a[:, 0].min()) / 1e9, 0)
    return {"workload": path.parts[-4], "cell": path.parent.name,
            "requests": len(rows), "clients": len(np.unique(a[:, 3])),
            "duration_s": span_s, "input_tok_median": np.median(a[:, 1]),
            "input_tok_p95": np.percentile(a[:, 1], 95),
            "output_tok_median": np.median(a[:, 2]), "output_tok_p95": np.percentile(a[:, 2], 95),
            "arrival_rate_rps": len(rows) / span_s if span_s else np.nan}

pilot_files = sorted(PILOT_DIR.glob("w*/r*_s*/trace.jsonl"))
pilot = pd.DataFrame([x for f in pilot_files if (x := inspect_trace(f)) is not None])
display(pilot.sort_values(["workload", "arrival_rate_rps"]).reset_index(drop=True))


Pilot files describe the offered load and request mix, but they do not reveal the simulator service limit on their own. The following cell uses direct pilot RR outputs if they have been added. Otherwise it reads the RR rows already summarized from the main suite.

In [ ]:
def read_request_csv(path, workload, rate, source):
    d = pd.read_csv(path)
    # Simulator request CSV times are nanoseconds.
    for col in ("queuing_delay", "latency", "end_time", "arrival"):
        if col in d:
            d[col] = pd.to_numeric(d[col], errors="coerce") / 1e6
    d["workload"], d["rate_rps"], d["source"] = workload, rate, source
    return d

def pilot_rr_results():
    found = []
    if not PILOT_RESULTS.exists():
        return found
    for p in PILOT_RESULTS.glob("w*/r*_s*/trace_policy_RR.csv"):
        m = re.search(r"(w[1-5])/r([0-9.]+)_s", str(p))
        if m:
            found.append(read_request_csv(p, m.group(1), float(m.group(2)), "pilot"))
    return found

pilot_runs = pilot_rr_results()
if pilot_runs:
    rr_requests = pd.concat(pilot_runs, ignore_index=True)
    print(f"Using {len(pilot_runs)} direct RR pilot cells")
else:
    if not SUMMARY_CSV.exists():
        raise FileNotFoundError(f"Neither pilot RR CSVs nor {SUMMARY_CSV} were found")
    summary = pd.read_csv(SUMMARY_CSV)
    rr_summary = summary.loc[summary.policy_name.eq("RR")].copy()
    rr_summary["rate_rps"] = rr_summary.apply(lambda r: float(r["rho"]) * {"w1":2,"w2":2,"w3":2,"w4":0.5,"w5":4}[r["workload"]], axis=1)
    print("No direct RR pilot CSVs found; using existing main-suite RR aggregate rows.")


## Find the RR knee

In [ ]:
if pilot_runs:
    records = []
    for (w, rate), d in rr_requests.groupby(["workload", "rate_rps"]):
        span = (d["end_time"].max() - d["arrival"].min()) / 1000 if {"end_time", "arrival"}.issubset(d.columns) else np.nan
        arrival_span = (d["arrival"].max() - d["arrival"].min()) / 1000 if "arrival" in d else np.nan
        records.append({"workload": w, "rate_rps": rate, "queue_p99_ms": d["queuing_delay"].quantile(.99),
                        "queue_p50_ms": d["queuing_delay"].median(),
                        "throughput_rps": len(d) / (span / 1000) if pd.notna(span) and span > 0 else np.nan,
                        "makespan_inflation": span / arrival_span if pd.notna(span) and arrival_span > 0 else np.nan,
                        "source": "pilot"})
    curve = pd.DataFrame(records)
else:
    curve = rr_summary.rename(columns={"queue_delay_p99_ms":"queue_p99_ms", "request_throughput":"throughput_rps"})
    curve["source"] = "existing main RR summary"
    # Map W4's aggregate request completion rate back to session start-rate scale for plotting only.

curve = curve.sort_values(["workload", "rate_rps"]).reset_index(drop=True)
show = [c for c in ["workload", "rate_rps", "queue_p50_ms", "queue_p99_ms", "throughput_rps", "makespan_inflation", "source"] if c in curve]
display(curve[show].round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for w, d in curve.groupby("workload"):
    axes[0].plot(d.rate_rps, d.queue_p99_ms, marker="o", label=w)
    axes[1].plot(d.rate_rps, d.throughput_rps, marker="o", label=w)
axes[0].set(xlabel="Offered rate (requests/s; W4 session starts/s)", ylabel="RR p99 queue delay (ms)", title="Queue delay")
axes[1].set(xlabel="Offered rate (requests/s; W4 session starts/s)", ylabel="Completed request throughput (req/s)", title="Completion throughput")
for ax in axes: ax.grid(alpha=.25)
axes[0].legend(ncol=2)
plt.tight_layout()


## Recommended `LAM_*` values

For each workload, select the largest tested rate before a queue-delay knee. A knee is flagged when p99 queue delay rises to at least 2× the preceding point or exceeds 1 second; throughput is shown alongside it to distinguish rising latency under saturation from a rate that the system continues to absorb. If no knee appears in the measured range, the function selects the highest tested rate and flags it as a lower bound on capacity.

In [ ]:
def choose_lam(d):
    d = d.sort_values("rate_rps").reset_index(drop=True)
    if len(d) < 2:
        return float(d.rate_rps.iloc[0]), "only one measured rate"
    base = max(float(d.queue_p99_ms.iloc[0]), 1.0)
    knee_idx = None
    for i in range(1, len(d)):
        prev, cur = float(d.queue_p99_ms.iloc[i-1]), float(d.queue_p99_ms.iloc[i])
        if cur >= 2 * max(prev, base) or cur >= 1000:
            knee_idx = i
            break
    if knee_idx is None:
        return float(d.rate_rps.iloc[-1]), "no knee observed; capacity is at least this rate"
    return float(d.rate_rps.iloc[knee_idx-1]), f"below queue-delay knee at {d.rate_rps.iloc[knee_idx]:g}"

recommendations = {}
for w, d in curve.groupby("workload"):
    recommendations[w] = choose_lam(d)
rec = pd.DataFrame([{"workload":w, "LAM_rate":rate, "selection":why} for w,(rate,why) in recommendations.items()])
display(rec)
print("\n# Add to your shell before generating main/scaling workloads:")
for w in sorted(recommendations):
    print(f"export LAM_{w.upper()}={recommendations[w][0]:g}")


## What makes the generated suite useful

Keep the workload mix fixed while sweeping the offered rate around each `LAM` (the generator’s default `LEVELS` are 0.5, 0.8, 1.0, 1.25, and 1.5). Use multiple seeds for the main comparison and retain the 7-seed tune split only for parameter tuning. W1/W3 scaling workloads vary client count separately from load. After direct pilot RR runs are available, rerun this notebook and update `EXPERIMENTS.md` if the selected rates change. The current documented values are 2, 2, 2, 0.5, and 4 respectively; the checked-in main-suite curves make these reasonable working estimates but do not substitute for direct pilot RR measurements.